## Heston CNN Model Calibration

In [ ]:
import pandas as pd
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import grad
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
from joblib import dump
import QuantLib as ql
from scipy.optimize import minimize
from py_vollib.black_scholes.implied_volatility import implied_volatility
import time

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load Data

In [ ]:
base_filename = 'heston_option'

In [ ]:
heston_data_list = list()
for i in range(0,100):
    filename = base_filename + str(i) + '.csv'
    if os.path.exists(filename):
        new_heston_data = pd.read_csv(filename, index_col=[0])
        heston_data_list.append(new_heston_data) 
    
heston_data = pd.concat(heston_data_list)
heston_data.reset_index(inplace=True)

In [ ]:
heston_data

In [ ]:
heston_data.drop(['index','price'], axis=1, inplace=True)

In [ ]:
eta_unique = heston_data['eta'].unique()

In [ ]:
eta_unique.shape

In [ ]:
input_lst = []
output_lst = []
for ieta in eta_unique:
    block = heston_data[heston_data['eta'] == ieta]
    eta = block['eta'].unique().item()
    rho = block['rho'].unique().item()
    lamda = block['lambda'].unique().item()
    vbar = block['vbar'].unique().item()
    V0 = block['V0'].unique().item()
    taus = block['tau'].unique()
    Ks = block['K'].unique()
    vols = np.zeros((len(Ks), len(taus)))
    for iK, K in enumerate(Ks):
        for itau, tau in enumerate(taus):
            taublock = block[block['K'] == K]
            Kblock = taublock[taublock['tau'] == tau]
            iv = Kblock['iv'].item()
            vols[iK, itau] = iv
    output_lst.append([eta, rho, lamda, vbar, V0])
    input_lst.append(vols)

In [ ]:
len(input_lst)

In [ ]:
X = np.array(input_lst)
y = np.array(output_lst)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.04, random_state=42)

In [ ]:
train_x = torch.Tensor(X_train).unsqueeze(dim=1).to(device)
train_y = torch.Tensor(y_train).to(device)

In [ ]:
test_x = torch.Tensor(X_test).unsqueeze(dim=1).to(device)
test_y = torch.Tensor(y_test).to(device)

In [ ]:
m = train_x.mean(0, keepdim=True)
s = train_x.std(0, unbiased=False, keepdim=True)
train_x -= m
train_x /= s
test_x -= m
test_x /= s

In [ ]:
batch_size = 256

In [ ]:
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, 
                                           shuffle=True, drop_last=True)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, 
                                          shuffle=False, drop_last=True)

## Train Model

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    grad_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        for batch_X, batch_y in train_loader:

            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)

        # Evaluation on the test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train loss: {train_loss:.6f}, \
                                   Test Loss: {test_loss:.6f}")
    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    return history

In [ ]:
class Swish(nn.Module):
    def __init__(self):
        super(Swish, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

In [ ]:
class HestonCNN(nn.Module):
    def __init__(self):
        super(HestonCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(8 * 5 * 4, 50)
        self.fc2 = nn.Linear(50, 5)
        self.swish = Swish()

    def forward(self, x):
        x = self.swish(self.conv1(x))
        x = self.pool(x)
        x = self.flatten(x)
        x = self.swish(self.fc1(x))
        x = self.fc2(x)  # Linear activation function in the output layer
        return x

In [ ]:
no_epochs = 1000
loss_fn = nn.MSELoss()

In [ ]:
hestoncnn = HestonCNN().to(device)
optimizer = optim.Adam(hestoncnn.parameters(), lr=0.001)
history = train_model(hestoncnn, train_loader, test_loader, 
                          loss_fn, optimizer, no_epochs)

In [ ]:
model_path = "hestocnn.pth"

In [ ]:
torch.save(hestoncnn.state_dict(), model_path)

### Plot Losses

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

train_loss_values = history['train_loss']
test_loss_values = history['test_loss']
ax.plot(train_loss_values, color='black', linestyle='-',label = 'training loss', )
ax.plot(test_loss_values, color='black', linestyle='--', label = 'test loss')

ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend()
ax.set_ylim(0, 0.1)
fig.savefig('HestonCNNLoss.png', dpi=300, bbox_inches='tight')

### Fit to an example Vol Surface

In [ ]:
test_idx = 2

In [ ]:
def hestonIV(S0, r, q, tau, K, V0, lamda, vbar, eta, rho):
    todaysDate = ql.Date(17, 2, 2024)
    ql.Settings.instance().evaluationDate = todaysDate
    maturity = int(tau * 365)
    settlementDate = todaysDate 
    day_count = ql.Actual365Fixed()
    rfr = ql.YieldTermStructureHandle(ql.FlatForward(settlementDate, r, day_count))
    div_yield = ql.YieldTermStructureHandle(ql.FlatForward(settlementDate, q, day_count))
    
    exercise = ql.EuropeanExercise(todaysDate + maturity)
    payoff = ql.PlainVanillaPayoff(ql.Option.Call, K)
    european_option = ql.VanillaOption(payoff, exercise)
        
    spot = ql.QuoteHandle(ql.SimpleQuote(S0))
    heston_process = ql.HestonProcess(rfr, div_yield, spot, V0, lamda, vbar, eta, rho)
    engine = ql.AnalyticHestonEngine(ql.HestonModel(heston_process), 1e-15, int(1e6))
    european_option.setPricingEngine(engine)
    price = european_option.NPV()
    iv = implied_volatility(price, S0, K, tau, r, 'c')
    return iv

In [ ]:
def objective_function(params, iv_surface):
    total_error = 0
    Ks = [0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5]
    taus = [0.1,0.3,0.6,0.9,1.2,1.5,1.8,2.0]
    for iK, K in enumerate(Ks):
        for itau, tau in enumerate(taus):
            market_iv = iv_surface[iK, itau]
            V0 = params[4]
            lamda = params[2]
            vbar = params[3]
            eta = params[0]
            rho = params[1]
        
        try:
            model_iv = hestonIV(1.0, 0.0, 0.0, tau, K, 
                               V0, lamda, vbar, eta, rho)
            error = (market_iv - model_iv) ** 2
        except Exception as e:
            print("Error for params: {}, mat: {}, strike: {}".format(params, tau, K))
            error = np.inf  # Assign a high error for failures
        total_error += error
    return total_error

In [ ]:
eta = 2.5
rho = -0.5
lamda = 5.0
vbar = 0.5
V0 = 0.5
S0 = 1.0
r = 0.0
q = 0.0
T = 0.14236824
strike = 0.5
param_lst = [eta, rho, lamda, vbar, V0]
initial_params = np.array(param_lst)

In [ ]:
hestonIV(S0, r, q, T, strike, V0, lamda, vbar, eta, rho)

In [ ]:
optimization_method = 'Nelder-Mead'

In [ ]:
bounds = [(0.0, 5.0), (-1.0, 0.0), (0.0, 10.0), (0.0, 1.0), (0.0, 1.0)]

In [ ]:
start_time = time.perf_counter() 
result = minimize(objective_function, initial_params, args=(X_test[test_idx],), 
                  bounds=bounds, method=optimization_method)
end_time = time.perf_counter()

In [ ]:
if result.success:
    fitted_params = result.x
    print("Optimization succeeded.")
    print(f"Fitted Heston parameters: {fitted_params}")
else:
    print("Optimization failed.")
    print(result.message)

In [ ]:
execution_time = (end_time - start_time) * 1000
print(f"Execution time: {execution_time} milliseconds")

In [ ]:
y_test[test_idx]

Rebuild the implied vol from fitted parameters

In [ ]:
Ks = [0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5]
taus = [0.1,0.3,0.6,0.9,1.2,1.5,1.8,2.0]

In [ ]:
feta = fitted_params[0]
frho = fitted_params[1]
flambda = fitted_params[2]
fvbar = fitted_params[3]
fV0 = fitted_params[4]

In [ ]:
fitted_iv = np.zeros((len(Ks), len(taus)))

In [ ]:
for iK, K in enumerate(Ks):
    for itau, tau in enumerate(taus):
        fitted_iv[iK, itau] = hestonIV(1.0, 0.0, 0.0, tau, K, 
                                      fV0, flambda, fvbar, feta, frho)

In [ ]:
example = test_x[test_idx].unsqueeze(dim=0)

In [ ]:
start_time = time.perf_counter() 
modelres = hestoncnn(example).detach().cpu().squeeze().numpy()
end_time = time.perf_counter() 

In [ ]:
modelres

In [ ]:
execution_time = (end_time - start_time) * 1000
print(f"Execution time: {execution_time} milliseconds")

Rebuild the implied vol from model parameters

In [ ]:
meta = modelres[0].item()
mrho = modelres[1].item()
mlambda = modelres[2].item()
mvbar = modelres[3].item()
mV0 = modelres[4].item()

In [ ]:
model_iv = np.zeros((len(Ks), len(taus)))

for iK, K in enumerate(Ks):
    for itau, tau in enumerate(taus):
        model_iv[iK, itau] = hestonIV(1.0, 0.0, 0.0, tau, K, 
                                      mV0, mlambda, mvbar, meta, mrho)

### Plot Smiles

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(14, 10))
axs = axs.flatten()

for i, maturity in enumerate(taus):
    axs[i].plot(Ks, X_test[test_idx].T[i], color='black', linestyle = ':', label='Ground Truth')
    axs[i].plot(Ks, fitted_iv.T[i], color='black', linestyle = '--', label='Fitted')
    axs[i].plot(Ks, model_iv.T[i], color='black', linestyle = '-.', label='DNN')
    axs[i].set_title(f'Maturity = {maturity} years')
    axs[i].set_xlabel('Relative Strike (K/S)')
    axs[i].set_ylabel('Implied Volatility')
    axs[i].legend()

fig.savefig('HestonCNNSmiles.png', dpi=300, bbox_inches='tight')